# 3일차 실습: 실시간 웹캠 필터 앱
---
## 오늘 목표
1~2일차에서 배운 모든 기술을 합쳐서 완성도 높은 앱을 만듭니다.

### 필터 앱 완성
- Step 1: 웹캠 연결 확인
- Step 2: 필터 함수 만들기 (6가지)
- Step 3: 메인 앱 실행
- Step 4: 트랙바로 파라미터 실시간 조절
- Step 5: 웹캠 영상 녹화 기능 추가

### 심화 실습
- Step 6: Day2 감정/포즈 파이프라인 실시간 연결
- Step 7: Streamlit 웹 앱 배포

### 키보드 조작
| 키 | 기능 |
|---|---|
| `1~6` | 필터 전환 |
| `r` | 녹화 시작/중지 |
| `s` | 현재 화면 저장 |
| `q` | 종료 |

## 0. 환경 설정 — OpenVINO 로컬 설치

처음 수업 시작 전에 한 번만 실행하면 됩니다.

### Python 3.11 설치 (3.13이면 필수)

**Windows**
1. https://www.python.org/downloads/release/python-3119/ 에서 다운로드
2. 설치 시 **'Add to PATH'** 체크 필수!

**Mac**
```bash
brew install python@3.11
```

### 가상환경 생성 및 패키지 설치

**Windows (터미널)**
```bash
py -3.11 -m venv cv_env
cv_env\Scripts\activate
pip install openvino openvino-dev opencv-python numpy matplotlib
```

**Mac (터미널)**
```bash
python3.11 -m venv cv_env
source cv_env/bin/activate
pip install openvino openvino-dev opencv-python numpy matplotlib
```

### VS Code에서 가상환경 선택
1. 우측 상단 '커널 선택' 클릭
2. '다른 커널 선택'
3. 'Python 환경'
4. '+ Python 환경 만들기'
5. 'venv' 클릭
6. 폴더 선택
7. '사용자 지정'
8. 3.11 버전 선택



##### 터미널 열어서 실행
```
pip install opencv-python openvino openvino-dev numpy matplotlib pillow
```


In [1]:
# 설치 확인 — 아래 셀을 실행해서 모두 OK 뜨면 준비 완료!
import sys
print(f'Python 버전: {sys.version}')

try:
    import cv2
    print(f'OpenCV: {cv2.__version__}  OK')
except ImportError:
    print('OpenCV 미설치 — pip install opencv-python')

try:
    from openvino.runtime import Core
    ie = Core()
    print(f'OpenVINO OK  사용 가능한 디바이스: {ie.available_devices}')
except ImportError:
    print('OpenVINO 미설치 — pip install openvino openvino-dev')

try:
    import numpy as np
    print(f'NumPy: {np.__version__}  OK')
except ImportError:
    print('NumPy 미설치 — pip install numpy')

try:
    import matplotlib
    print(f'Matplotlib: {matplotlib.__version__}  OK')
except ImportError:
    print('Matplotlib 미설치 — pip install matplotlib')


Python 버전: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
OpenCV: 4.13.0  OK
OpenVINO OK  사용 가능한 디바이스: ['CPU', 'GPU']
NumPy: 2.4.3  OK
Matplotlib: 3.10.8  OK


In [2]:
from PIL import ImageFont, ImageDraw, Image
import numpy as np
import cv2

# 한글 폰트 로드 (Windows 기본 폰트)
font_path = "C:/Windows/Fonts/malgun.ttf"   # 맑은 고딕
font = ImageFont.truetype(font_path, 24)     # 크기 24

def put_korean_text(img, text, position, color=(0, 255, 255)):
    """OpenCV 이미지에 한글 텍스트 쓰기"""
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    draw.text(position, text, font=font, fill=color)
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

## Step 1 — 웹캠 연결 확인

In [5]:
import cv2
import numpy as np
from datetime import datetime

camera = cv2.VideoCapture(0)
if not camera.isOpened():
    print('❌ 웹캠을 열 수 없습니다.')
else:
    ret, frame = camera.read()
    print(f'✅ 웹캠 연결 성공! 해상도: {frame.shape[1]}x{frame.shape[0]}')
    camera.release()

✅ 웹캠 연결 성공! 해상도: 640x480


## Step 2 — 필터 함수 만들기

각 필터를 함수로 분리해서 깔끔하게 관리합니다.

In [6]:
# 각 필터를 함수로 정의

def apply_original(frame):
    return frame                                              # 원본 그대로

def apply_grayscale(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)            # 회색조 → 다시 BGR

def apply_blur(frame):
    return cv2.GaussianBlur(frame, (21, 21), 100)             # Gaussian 블러

def apply_canny(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)             # 노이즈 제거
    edges = cv2.Canny(blurred, 50, 150)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

def apply_contour(frame):
    result = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    large = [c for c in contours if cv2.contourArea(c) > 1000]
    cv2.drawContours(result, large, -1, (0, 255, 0), 2)
    for c in large:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 0, 255), 1)
    cv2.putText(result, f'Objects: {len(large)}', (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return result

def apply_color_mask(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([100, 50, 50]), np.array([130, 255, 255]))
    return cv2.bitwise_and(frame, frame, mask=mask)

# 필터 딕셔너리: 키코드 → (함수, 이름)
FILTERS = {
    ord('1'): (apply_original,   '1: 원본'),
    ord('2'): (apply_grayscale,  '2: 회색조'),
    ord('3'): (apply_blur,       '3: 블러'),
    ord('4'): (apply_canny,      '4: Canny 엣지'),
    ord('5'): (apply_contour,    '5: Contour'),
    ord('6'): (apply_color_mask, '6: 파란색 마스크'),
}
print('✅ 필터 함수 정의 완료!')
print('사용 가능한 필터:', [v[1] for v in FILTERS.values()])

✅ 필터 함수 정의 완료!
사용 가능한 필터: ['1: 원본', '2: 회색조', '3: 블러', '4: Canny 엣지', '5: Contour', '6: 파란색 마스크']


## Step 3 — 메인 앱 실행

모든 필터를 합쳐서 인터랙티브 앱을 실행합니다.

In [7]:
# 실시간 웹캠 필터 앱
camera = cv2.VideoCapture(0)
current_filter_func = apply_original
current_filter_name = '1: 원본'
saved_count = 0

print('앱 시작! 키 조작: 1~6 필터 전환 / s 저장 / q 종료')

while True:
    ret, frame = camera.read()
    if not ret:
        break

    filtered = current_filter_func(frame)                       # 필터 적용

    # 정보 오버레이
    timestamp = datetime.now().strftime('%H:%M:%S')
    cv2.putText(filtered, f'Filter: {current_filter_name}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(filtered, timestamp, (10, filtered.shape[0] - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.putText(filtered, 's:저장  q:종료', (filtered.shape[1]-170, filtered.shape[0]-15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    cv2.imshow('실시간 필터 앱 (1~6: 필터 전환)', filtered)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('s'):
        filename = f"capture_{datetime.now().strftime('%H%M%S')}.png"
        cv2.imwrite(filename, filtered)
        saved_count += 1
        print(f'저장됨: {filename} (총 {saved_count}장)')
    elif key in FILTERS:
        current_filter_func, current_filter_name = FILTERS[key]
        print(f'필터 변경: {current_filter_name}')

camera.release()
cv2.destroyAllWindows()
print(f'앱 종료. 총 {saved_count}장 저장됨.')

앱 시작! 키 조작: 1~6 필터 전환 / s 저장 / q 종료
필터 변경: 1: 원본
필터 변경: 2: 회색조
필터 변경: 3: 블러
필터 변경: 4: Canny 엣지
필터 변경: 5: Contour
필터 변경: 6: 파란색 마스크
앱 종료. 총 0장 저장됨.


## Step 4 — 트랙바로 파라미터 실시간 조절

`cv2.createTrackbar()`를 사용하면 슬라이더로 값을 바꾸면서 결과를 실시간으로 확인할 수 있습니다.

Canny 엣지의 threshold 값을 트랙바로 조절해봅니다.

> 💡 트랙바는 `cv2.namedWindow()`로 만든 창에만 붙일 수 있어요.

In [ ]:
# 트랙바로 Canny threshold 실시간 조절
camera = cv2.VideoCapture(0)

# 트랙바 창 생성
cv2.namedWindow('Canny 트랙바 (q: 종료)')
cv2.createTrackbar('Threshold1', 'Canny 트랙바 (q: 종료)', 50,  300, lambda x: None)
cv2.createTrackbar('Threshold2', 'Canny 트랙바 (q: 종료)', 150, 600, lambda x: None)
cv2.createTrackbar('Blur',       'Canny 트랙바 (q: 종료)', 5,   21,  lambda x: None)

while True:
    ret, frame = camera.read()
    if not ret:
        break

    # 트랙바 값 읽기
    t1   = cv2.getTrackbarPos('Threshold1', 'Canny 트랙바 (q: 종료)')
    t2   = cv2.getTrackbarPos('Threshold2', 'Canny 트랙바 (q: 종료)')
    blur = cv2.getTrackbarPos('Blur',       'Canny 트랙바 (q: 종료)')

    # blur 값은 홀수여야 함
    blur = blur if blur % 2 == 1 else blur + 1
    blur = max(1, blur)

    # Canny 적용
    gray    = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (blur, blur), 0)
    edges   = cv2.Canny(blurred, t1, t2)
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # 파라미터 값 화면에 표시
    cv2.putText(edges_bgr, f'T1:{t1}  T2:{t2}  Blur:{blur}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    # 원본과 나란히 표시
    combined = np.hstack([frame, edges_bgr])
    cv2.imshow('Canny 트랙바 (q: 종료)', combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

camera.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

### 🎯 GenAI 활용 — 트랙바 확장

---
**프롬프트:**
```
Python OpenCV 트랙바 앱에 다음을 추가해줘:

현재 Canny 엣지에 트랙바가 있는데,
트랙바로 선택 가능한 필터 모드를 추가해줘:
- Mode 0: Canny 엣지
- Mode 1: Gaussian 블러 (커널 크기 트랙바)
- Mode 2: HSV 마스크 (H 범위 트랙바 2개)

'Mode' 트랙바 값에 따라 다른 필터가 적용되고
현재 모드 이름이 화면 상단에 표시되도록 해줘.
```
---

In [ ]:
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
# your code here

# 트랙바로 Canny threshold 실시간 조절
camera = cv2.VideoCapture(0)

# 트랙바 창 생성
cv2.namedWindow('Canny 트랙바 (q: 종료)')
cv2.createTrackbar('Threshold1', 'Canny 트랙바 (q: 종료)', 50,  300, lambda x: None)
cv2.createTrackbar('Threshold2', 'Canny 트랙바 (q: 종료)', 150, 600, lambda x: None)
cv2.createTrackbar('Blur',       'Canny 트랙바 (q: 종료)', 5,   21,  lambda x: None)

while True:
    ret, frame = camera.read()
    if not ret:
        break

    # 트랙바 값 읽기
    t1   = cv2.getTrackbarPos('Threshold1', 'Canny 트랙바 (q: 종료)')
    t2   = cv2.getTrackbarPos('Threshold2', 'Canny 트랙바 (q: 종료)')
    blur = cv2.getTrackbarPos('Blur',       'Canny 트랙바 (q: 종료)')

    # blur 값은 홀수여야 함
    blur = blur if blur % 2 == 1 else blur + 1
    blur = max(1, blur)

    # Canny 적용
    gray    = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (blur, blur), 0)
    edges   = cv2.Canny(blurred, t1, t2)
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # 파라미터 값 화면에 표시
    cv2.putText(edges_bgr, f'T1:{t1}  T2:{t2}  Blur:{blur}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    # 원본과 나란히 표시
    combined = np.hstack([frame, edges_bgr])
    cv2.imshow('Canny 트랙바 (q: 종료)', combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

import cv2
import numpy as np

# 트랙바 콜백을 위한 더미 함수
def nothing(x):
    pass

# 웹캠 연결
camera = cv2.VideoCapture(0)

# 창 이름 설정
window_name = 'Interactive Filter App (q: 종료)'
cv2.namedWindow(window_name)

# 1. 트랙바 생성
# [모드 선택 트랙바] 0: Canny, 1: Blur, 2: HSV
cv2.createTrackbar('Mode', window_name, 0, 2, nothing)

# [Canny 파라미터]
cv2.createTrackbar('Canny T1', window_name, 50, 300, nothing)
cv2.createTrackbar('Canny T2', window_name, 150, 600, nothing)

# [Blur 커널 크기] (Canny의 노이즈 제거용, Blur 모드의 커널용 공통 사용)
cv2.createTrackbar('Blur Size', window_name, 5, 31, nothing)

# [HSV 파라미터] H(색조) 범위 설정
cv2.createTrackbar('H Min', window_name, 0, 179, nothing)
cv2.createTrackbar('H Max', window_name, 179, 179, nothing)

# 모드별 이름 지정
mode_names = ['Mode 0: Canny Edge', 'Mode 1: Gaussian Blur', 'Mode 2: HSV Mask']

print("앱이 실행되었습니다. 'q'를 누르면 종료됩니다.")

while True:
    ret, frame = camera.read()
    if not ret:
        break

    # 2. 트랙바 현재 값 읽어오기
    mode = cv2.getTrackbarPos('Mode', window_name)
    t1 = cv2.getTrackbarPos('Canny T1', window_name)
    t2 = cv2.getTrackbarPos('Canny T2', window_name)
    blur_val = cv2.getTrackbarPos('Blur Size', window_name)
    h_min = cv2.getTrackbarPos('H Min', window_name)
    h_max = cv2.getTrackbarPos('H Max', window_name)

    # 블러 커널 크기는 무조건 1 이상의 홀수여야 함
    ksize = blur_val if blur_val % 2 == 1 else blur_val + 1
    ksize = max(1, ksize)

    result = frame.copy()
    param_text = ""

    # 3. Mode 트랙바 값에 따른 필터 적용
    if mode == 0:
        # --- Mode 0: Canny 엣지 ---
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (ksize, ksize), 0)
        edges = cv2.Canny(blurred, t1, t2)
        # 원본(BGR)과 합치기 위해 채널 수 맞추기
        result = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
        param_text = f'T1:{t1} T2:{t2} Blur:{ksize}'

    elif mode == 1:
        # --- Mode 1: Gaussian 블러 ---
        result = cv2.GaussianBlur(frame, (ksize, ksize), 0)
        param_text = f'Kernel Size: {ksize}'

    elif mode == 2:
        # --- Mode 2: HSV 마스크 ---
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        # S(채도), V(명도)는 중간값 이상만 추출하도록 고정하여 색상을 더 잘 잡게 설정
        lower_bound = np.array([h_min, 50, 50])
        upper_bound = np.array([h_max, 255, 255])
        
        mask = cv2.inRange(hsv, lower_bound, upper_bound)
        result = cv2.bitwise_and(frame, frame, mask=mask)
        param_text = f'H Range: {h_min} - {h_max}'

    # 4. 화면 상단에 텍스트 정보 표시
    # 현재 모드 이름 표시 (노란색)
    cv2.putText(result, mode_names[mode], (10, 35), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
    
    # 현재 적용 중인 파라미터 값 표시 (초록색)
    cv2.putText(result, param_text, (10, 70), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # 5. 원본 이미지와 결과 이미지를 가로로 나란히(hstack) 결합하여 출력
    combined = np.hstack([frame, result])
    cv2.imshow(window_name, combined)

    # 'q' 키를 누르면 루프 탈출
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    # 's' 키를 누르면 현재 필터 적용 화면 저장
    elif key == ord('s'):
        # 파일명 생성 (예: capture_142530.png)
        filename = f"capture_{datetime.now().strftime('%H%M%S')}.png"
        
        # 필터가 적용된 결과 화면(result)만 저장 
        # (만약 원본과 나란히 있는 화면을 저장하고 싶다면 result 대신 combined 입력)
        cv2.imwrite(filename, result)
        
        saved_count += 1
        print(f"이미지 저장 완료: {filename} (총 {saved_count}장)")

# 자원 해제
camera.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

: 

## Step 5 — 웹캠 영상 녹화 기능 추가

`cv2.VideoWriter`로 필터가 적용된 영상을 파일로 저장할 수 있습니다.

```
r 키 → 녹화 시작  →  파일에 프레임 저장  →  r 키 → 녹화 중지
```

In [17]:
# 녹화 기능이 추가된 웹캠 필터 앱
from datetime import datetime

camera  = cv2.VideoCapture(0)
ret, frame = camera.read()
h, w = frame.shape[:2]

# VideoWriter 설정
fourcc    = cv2.VideoWriter_fourcc(*'mp4v')  # mp4 포맷
writer    = None
recording = False

current_filter_func = apply_original
current_filter_name = '1: 원본'
saved_count = 0

print('앱 시작!  r: 녹화시작/중지  s: 캡처  q: 종료')

while True:
    ret, frame = camera.read()
    if not ret:
        break

    filtered = current_filter_func(frame)

    # 녹화 중이면 프레임 저장
    if recording and writer is not None:
        writer.write(filtered)

    # 정보 오버레이
    timestamp = datetime.now().strftime('%H:%M:%S')
    rec_text  = '● REC' if recording else ''
    cv2.putText(filtered, f'Filter: {current_filter_name}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(filtered, timestamp, (10, filtered.shape[0]-15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    if recording:
        cv2.putText(filtered, 'REC', (filtered.shape[1]-80, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

    cv2.imshow('필터 앱 with 녹화 (r:녹화  s:저장  q:종료)', filtered)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('r'):   # 녹화 시작/중지
        if not recording:
            fname   = f'recording_{datetime.now().strftime("%H%M%S")}.mp4'
            writer  = cv2.VideoWriter(fname, fourcc, 20.0, (w, h))
            recording = True
            print(f'녹화 시작: {fname}')
        else:
            writer.release()
            writer    = None
            recording = False
            print('녹화 중지')
    elif key == ord('s'):   # 화면 저장
        fname = f'capture_{datetime.now().strftime("%H%M%S")}.png'
        cv2.imwrite(fname, filtered)
        saved_count += 1
        print(f'저장: {fname}')
    elif key in FILTERS:
        current_filter_func, current_filter_name = FILTERS[key]

if writer:
    writer.release()
camera.release()
cv2.destroyAllWindows()
print(f'종료. 캡처 {saved_count}장')


앱 시작!  r: 녹화시작/중지  s: 캡처  q: 종료
종료. 캡처 0장


## 🎯 도전 과제 1 — GenAI 활용: 새 필터 추가

---
**프롬프트:**
```
Python OpenCV로 웹캠 영상에 적용할 새로운 필터 함수를 만들어줘.
다음 3가지 중 하나를 골라서 구현해줘:

A) 세피아 필터: 이미지를 빈티지 세피아 톤으로 변환 (numpy 행렬 변환 사용)
B) 카툰 필터: 만화처럼 보이게 변환 (bilateralFilter + Canny 엣지 합치기)
C) 야간 투시 필터: 초록색 야간 투시경 효과 (회색조를 초록 채널에만 적용)

함수 이름은 apply_새이름(frame) 형태로, frame 입력 받아 BGR 이미지 반환.
```
만든 함수를 FILTERS 딕셔너리에 ord('7') 키로 추가해서 테스트하세요!
---

In [10]:
# 여기에 GenAI가 생성한 새 필터를 붙여넣으세요
# your code here

# 아래처럼 FILTERS에 추가 후 앱을 다시 실행하세요:
# FILTERS[ord('7')] = (apply_새이름, '7: 새필터이름')

import cv2
import numpy as np

# A) 세피아 필터 (Sepia Filter)
def apply_sepia(frame):
    # OpenCV는 BGR 순서이므로, 이에 맞춘 세피아 변환 행렬 적용
    kernel = np.array([
        [0.131, 0.534, 0.272],
        [0.168, 0.686, 0.349],
        [0.189, 0.769, 0.393]
    ])
    # 행렬 곱셈을 통해 색상 변환
    sepia = cv2.transform(frame, kernel)
    # 값이 255를 초과하지 않도록 제한 후 uint8 타입으로 변환
    sepia = np.clip(sepia, 0, 255).astype(np.uint8)
    return sepia

# B) 카툰 필터 (Cartoon Filter)
def apply_cartoon(frame):
    # 1. 색상 부드럽게 만들기 (Bilateral Filter: 경계선은 유지하면서 노이즈/질감 감소)
    color = cv2.bilateralFilter(frame, d=9, sigmaColor=75, sigmaSpace=75)
    
    # 2. 흑백 변환 후 Canny 엣지 검출
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 75, 100)
    
    # 3. 엣지 반전 (흰색 배경에 검은색 선으로 만들기)
    edges = cv2.bitwise_not(edges)
    
    # 4. 엣지를 3채널(BGR)로 변환하여 합성 준비
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    
    # 5. 색상 이미지와 엣지 이미지 합성
    cartoon = cv2.bitwise_and(color, edges_bgr)
    return cartoon

# C) 야간 투시 필터 (Night Vision Filter)
def apply_night_vision(frame):
    # 1. 원본을 흑백으로 변환 (밝기 정보만 추출)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 2. 원본과 동일한 크기의 빈(검은색) 이미지 생성
    night_vision = np.zeros_like(frame)
    
    # 3. BGR 채널 중 G(초록색, 인덱스 1) 채널에만 흑백 밝기 값 덮어쓰기
    night_vision[:, :, 1] = gray
    
    return night_vision


# 기존 FILTERS 딕셔너리에 새로운 필터 3개 추가
# (참고: 이전에 정의한 FILTERS 변수가 메모리에 로드되어 있어야 합니다)
FILTERS[ord('7')] = (apply_sepia, '7: 세피아')
FILTERS[ord('8')] = (apply_cartoon, '8: 카툰')
FILTERS[ord('9')] = (apply_night_vision, '9: 야간 투시')

print('✅ 세피아, 카툰, 야간 투시 필터가 성공적으로 추가되었습니다!')
print('현재 사용 가능한 필터:', [v[1] for v in FILTERS.values()])

✅ 세피아, 카툰, 야간 투시 필터가 성공적으로 추가되었습니다!
현재 사용 가능한 필터: ['1: 원본', '2: 회색조', '3: 블러', '4: Canny 엣지', '5: Contour', '6: 파란색 마스크', '7: 세피아', '8: 카툰', '9: 야간 투시']


In [20]:
import cv2
import numpy as np
from datetime import datetime

# --- [필터 함수 정의] ---

def apply_original(frame):
    return frame

def apply_grayscale(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

def apply_blur(frame):
    return cv2.GaussianBlur(frame, (21, 21), 0)

def apply_canny(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

def apply_contour(frame):
    res = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    large = [c for c in contours if cv2.contourArea(c) > 1000]
    cv2.drawContours(res, large, -1, (0, 255, 0), 2)
    cv2.putText(res, f'Objects: {len(large)}', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return res

def apply_color_mask(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([100, 50, 50]), np.array([130, 255, 255]))
    return cv2.bitwise_and(frame, frame, mask=mask)

# 신규 추가 필터 3종 (7, 8, 9)
def apply_sepia(frame):
    kernel = np.array([[0.272, 0.534, 0.131],
                       [0.349, 0.686, 0.168],
                       [0.393, 0.769, 0.189]])
    sepia = cv2.transform(frame, kernel)
    return np.clip(sepia, 0, 255).astype(np.uint8)

def apply_cartoon(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.adaptiveThreshold(cv2.medianBlur(gray, 7), 255, 
                                  cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 10)
    color = cv2.bilateralFilter(frame, 9, 250, 250)
    return cv2.bitwise_and(color, color, mask=edges)

def apply_night_vision(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    noise = np.random.randint(0, 40, (frame.shape[0], frame.shape[1]), dtype='uint8')
    gray = cv2.add(gray, noise)
    nv = np.zeros_like(frame)
    nv[:, :, 1] = gray # Green 채널에 주입
    return nv

# 10: 픽셀아트 필터
def apply_pixelation(frame):
    # 1. 축소할 크기 결정 (원본의 1/20)
    h, w = frame.shape[:2]
    temp = cv2.resize(frame, (w//20, h//20), interpolation=cv2.INTER_LINEAR)
    # 2. 다시 확대 (INTER_NEAREST를 써야 픽셀이 깨진 느낌이 유지됨)
    return cv2.resize(temp, (w, h), interpolation=cv2.INTER_NEAREST)

# 11: 연필 스케치 필터
def apply_sketch(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    inv_gray = 255 - gray
    blurred = cv2.GaussianBlur(inv_gray, (21, 21), 0)
    sketch = cv2.divide(gray, 255 - blurred, scale=256)
    return cv2.cvtColor(sketch, cv2.COLOR_GRAY2BGR)

prev_frame = None  # 전역 변수 초기화

def apply_ghost(frame):
    global prev_frame
    if prev_frame is None:
        prev_frame = frame.copy()
        return frame
    
    # 이전 프레임과 현재 프레임을 합성 (0.85와 0.15 비중)
    ghost = cv2.addWeighted(prev_frame, 0.85, frame, 0.15, 0)
    prev_frame = ghost.copy()  # 현재 합상된 결과를 다음 프레임의 '이전 프레임'으로 저장
    return ghost

def apply_emboss(frame):
    # 엠보싱 커널 정의
    kernel = np.array([[-2, -1, 0],
                       [-1,  1, 1],
                       [ 0,  1, 2]])
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # 필터 적용 후 밝기 보정(+128)
    emboss = cv2.filter2D(gray, -1, kernel) + 128
    return cv2.cvtColor(emboss, cv2.COLOR_GRAY2BGR)

def apply_fisheye(frame):
    h, w = frame.shape[:2]
    
    # 1. 기본 그리드 생성 (float32 필수)
    map_y, map_x = np.indices((h, w), dtype=np.float32)
    
    # 2. 중앙으로부터의 거리 및 각도 계산
    center_x, center_y = w / 2, h / 2
    dx = map_x - center_x
    dy = map_y - center_y
    r = np.sqrt(dx**2 + dy**2)
    
    # 3. 왜곡 공식 적용 (가운데를 볼록하게)
    # r_max는 화면 끝까지의 거리
    r_max = np.sqrt(center_x**2 + center_y**2)
    theta = np.arctan2(dy, dx)
    
    # 새로운 반지름 계산 (r^2 연산으로 중앙부 확대 효과)
    new_r = (r**2) / r_max 
    
    # 4. 새로운 좌표 맵 생성
    new_x = center_x + new_r * np.cos(theta)
    new_y = center_y + new_r * np.sin(theta)
    
    # 5. 데이터 타입 재확인 및 리맵핑 (오류 방지 핵심)
    return cv2.remap(frame, new_x.astype(np.float32), new_y.astype(np.float32), cv2.INTER_LINEAR)


# --- [설정 및 매핑] ---

FILTERS = {
    ord('1'): (apply_original,   '1: Original'),
    ord('2'): (apply_grayscale,  '2: Gray'),
    ord('3'): (apply_blur,       '3: Blur'),
    ord('4'): (apply_canny,      '4: Canny'),
    ord('5'): (apply_contour,    '5: Contour'),
    ord('6'): (apply_color_mask, '6: Blue Mask'),
    ord('7'): (apply_sepia,      '7: Sepia'),
    ord('8'): (apply_cartoon,    '8: Cartoon'),
    ord('9'): (apply_night_vision, '9: Night Vision'),
    ord('0'): (apply_pixelation, '0: Pixel Art'),
    ord('-'): (apply_sketch,    '-: Pencil Sketch'),
    ord('='): (apply_ghost,   '=: Ghost Effect'),
    ord(']'): (apply_emboss, ']: Emboss'),
    ord('['): (apply_fisheye, '[: Fish Eye'),
}

camera = cv2.VideoCapture(0)
ret, frame = camera.read()
if not ret: exit()

h, w = frame.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = None
recording = False
current_filter_func = apply_original
current_filter_name = '1: Original'
saved_count = 0

print("🚀 앱 시작! (1~9: 필터 변경, r: 녹화, s: 캡처, q: 종료)")

# --- [메인 루프] ---

while True:
    ret, frame = camera.read()
    if not ret: break

    # 필터 적용
    filtered = current_filter_func(frame)

    # 녹화 로직
    if recording and writer is not None:
        writer.write(filtered)

    # UI 오버레이
    info = f'Filter: {current_filter_name}'
    cv2.putText(filtered, info, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    
    if recording:
        cv2.circle(filtered, (w - 30, 30), 10, (0, 0, 255), -1) # 빨간 점
        cv2.putText(filtered, 'REC', (w - 80, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.imshow('Smart Filter Cam', filtered)

    # 키 입력 처리
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('r'):
        if not recording:
            fname = f'rec_{datetime.now().strftime("%H%M%S")}.mp4'
            writer = cv2.VideoWriter(fname, fourcc, 20.0, (w, h))
            recording = True
            print(f"🔴 녹화 시작: {fname}")
        else:
            writer.release()
            writer = None
            recording = False
            print("⚪ 녹화 중지 및 저장 완료")
    elif key == ord('s'):
        fname = f'img_{datetime.now().strftime("%H%M%S")}.png'
        cv2.imwrite(fname, filtered)
        saved_count += 1
        print(f"📸 캡처 저장: {fname}")
    elif key in FILTERS:
        current_filter_func, current_filter_name = FILTERS[key]

# 리소스 해제
if writer: writer.release()
camera.release()
cv2.destroyAllWindows()

🚀 앱 시작! (1~9: 필터 변경, r: 녹화, s: 캡처, q: 종료)


In [19]:
import cv2
import numpy as np

# [레시피 1] 열화상 카메라 필터 (Thermal Heatmap)
def apply_thermal(frame):
    # 1. 영상을 흑백으로 변환하여 밝기 정보만 가져옵니다.
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 2. OpenCV의 기본 컬러맵(JET)을 씌워 밝은 곳은 붉게, 어두운 곳은 푸르게 만듭니다.
    thermal = cv2.applyColorMap(gray, cv2.COLORMAP_JET)
    return thermal

# [레시피 2] 사이버펑크 글리치 필터 (Glitch Art)
def apply_glitch(frame):
    glitch = frame.copy()
    h, w = glitch.shape[:2]
    
    # 1. 무작위 가로줄 노이즈 생성 (화면이 지지직거리는 효과)
    for _ in range(5): # 5개의 무작위 줄 생성
        y1 = np.random.randint(0, h - 20)
        y2 = y1 + np.random.randint(5, 20)
        # 해당 영역의 색상을 반전시킵니다.
        glitch[y1:y2, :] = cv2.bitwise_not(glitch[y1:y2, :])
        
    # 2. RGB 채널 분리 후 좌우로 어긋나게 밀어버리기
    b, g, r = cv2.split(glitch)
    # 빨간색(R) 채널을 오른쪽으로 15픽셀 밀기
    r = np.roll(r, 15, axis=1) 
    # 파란색(B) 채널을 왼쪽으로 15픽셀 밀기
    b = np.roll(b, -15, axis=1) 
    
    # 3. 분리했던 채널을 다시 하나로 합칩니다.
    return cv2.merge((b, g, r))


# [메뉴판 업데이트] FILTERS 딕셔너리에 단축키 알파벳 'a', 'b'로 추가
FILTERS[ord('a')] = (apply_thermal, 'A: 열화상 카메라')
FILTERS[ord('b')] = (apply_glitch, 'B: 사이버펑크 글리치')

print('✅ 열화상 카메라, 글리치 필터가 추가되었습니다!')

✅ 열화상 카메라, 글리치 필터가 추가되었습니다!


## 🎯 도전 과제 2 — GenAI 활용: 화면 분할 비교

---
**프롬프트:**
```
Python OpenCV 웹캠 앱에서 화면을 좌우로 분할해서
왼쪽에는 원본, 오른쪽에는 Canny 엣지 결과를 동시에 표시하는 코드를 작성해줘.

조건:
- np.hstack()으로 두 이미지를 가로로 붙이기
- 각 화면 왼쪽 상단에 '원본', 'Canny 엣지' 텍스트 표시
- 'q' 키로 종료
```
---

In [ ]:
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
# your code here

---
## Step 6 — Day2 감정/포즈 파이프라인 실시간 연결

Day2에서 만든 `analyze_image()` 를 웹캠 실시간 스트림에 연결합니다.

```
웹캠 프레임
    ├→ 얼굴 검출  →  감정 인식  →  결과 오버레이
    └→ 관절 추정  →  스틱맨
```

> ⚠️ 3개 모델을 매 프레임마다 실행하면 느릴 수 있어요.  
> N프레임마다 한 번씩 분석하는 방식으로 속도를 조절합니다.

omz_downloader --name face-detection-adas-0001 --output_dir ./models

omz_downloader --name emotions-recognition-retail-0003 --output_dir ./models

omz_downloader --name human-pose-estimation-0001 --output_dir ./models

In [3]:
from openvino.runtime import Core
import numpy as np

# Day2에서 만든 모델 로드 (로컬 경로)
MODEL_DIR = r'C:\Users\ComHolic\Desktop\인텔교육장\0318_에이전트만들기\models'
ie = Core()

face_compiled    = ie.compile_model(ie.read_model(f'{MODEL_DIR}/intel/face-detection-adas-0001/FP32/face-detection-adas-0001.xml'), 'CPU')
emotion_compiled = ie.compile_model(ie.read_model(f'{MODEL_DIR}/intel/emotions-recognition-retail-0003/FP32/emotions-recognition-retail-0003.xml'), 'CPU')
pose_compiled    = ie.compile_model(ie.read_model(f'{MODEL_DIR}/intel/human-pose-estimation-0001/FP32/human-pose-estimation-0001.xml'), 'CPU')

EMOTIONS   = ['neutral', 'happy', 'sad', 'surprise', 'anger']
EMOTION_COLOR = {'happy':(0,255,0),'neutral':(255,255,0),'sad':(255,100,0),'surprise':(0,200,255),'anger':(0,0,255)}
POSE_PAIRS = [(1,2),(1,5),(2,3),(3,4),(5,6),(6,7),(1,8),(8,9),(9,10),(1,11),(11,12),(12,13),(1,0),(0,14),(14,16),(0,15),(15,17)]

print('모델 로드 완료!')


모델 로드 완료!


In [5]:
from PIL import ImageFont, ImageDraw, Image
import numpy as np
import cv2

# 한글 폰트 설정
font_path = "C:/Windows/Fonts/malgun.ttf"
font_small  = ImageFont.truetype(font_path, 20)
font_medium = ImageFont.truetype(font_path, 26)

face_input_h = face_compiled.input(0).shape[2]
face_input_w = face_compiled.input(0).shape[3]

def detect_faces(frame, threshold=0.5):
    h, w   = frame.shape[:2]
    f_in   = cv2.resize(frame, (face_input_w, face_input_h))
    f_in   = np.expand_dims(f_in.transpose(2,0,1), 0).astype(np.float32)
    dets   = face_compiled([f_in])[face_compiled.output(0)]
    result = []
    for det in dets[0][0]:
        if det[2] > threshold:
            x1,y1 = max(0,int(det[3]*w)), max(0,int(det[4]*h))
            x2,y2 = min(w,int(det[5]*w)), min(h,int(det[6]*h))
            result.append((x1,y1,x2,y2,float(det[2])))
    return result

def put_korean_text(img, text, position, color=(0, 255, 255), font=None):
    if font is None:
        font = font_medium
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    draw    = ImageDraw.Draw(img_pil)
    r, g, b = color[2], color[1], color[0]
    draw.text(position, text, font=font, fill=(r, g, b))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

# 실시간 감정 + 포즈 분석 앱
camera      = cv2.VideoCapture(0)
frame_skip  = 5
frame_count = 0
last_result = None

def analyze_frame(frame):
    result = frame.copy()
    h, w   = frame.shape[:2]

    # 1. 얼굴 검출
    detections = detect_faces(frame)

    for (x1, y1, x2, y2, conf) in detections:
        # 2. 감정 인식
        roi = frame[y1:y2, x1:x2]
        if roi.size > 0:
            e_in  = np.expand_dims(cv2.resize(roi, (64, 64)).transpose(2, 0, 1), 0).astype(np.float32)
            probs = emotion_compiled([e_in])[emotion_compiled.output(0)][0].flatten()
            emo   = EMOTIONS[int(np.argmax(probs))]
            color = EMOTION_COLOR.get(emo, (255, 255, 255))

            cv2.rectangle(result, (x1, y1), (x2, y2), color, 2)
            label_y = max(0, y1 - 35)
            result  = put_korean_text(result, f'{emo} {probs.max():.0%}', (x1, label_y), color, font_medium)

    # 3. 관절 포즈 추정
    pi      = pose_compiled.input(0)
    ph, pw  = pi.shape[2], pi.shape[3]
    p_in    = np.expand_dims(cv2.resize(frame, (pw, ph)).transpose(2, 0, 1), 0).astype(np.float32)
    heatmaps = pose_compiled([p_in])[pose_compiled.output(1)]

    keypoints = []
    for i in range(18):
        hm = heatmaps[0, i]
        _, conf, _, pt = cv2.minMaxLoc(hm)
        keypoints.append((int(pt[0]*w/hm.shape[1]), int(pt[1]*h/hm.shape[0]), float(conf)))

    for (p1, p2) in POSE_PAIRS:
        x1, y1, c1 = keypoints[p1]
        x2, y2, c2 = keypoints[p2]
        if c1 > 0.1 and c2 > 0.1:
            cv2.line(result, (x1, y1), (x2, y2), (0, 255, 255), 2)
    for (x, y, c) in keypoints:
        if c > 0.1:
            cv2.circle(result, (x, y), 4, (0, 200, 255), -1)

    return result

print('앱 시작!  q: 종료')
print(f'frame_skip={frame_skip} — 위/아래 방향키로 조절')

while True:
    ret, frame = camera.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % frame_skip == 0:
        last_result = analyze_frame(frame)

    display = last_result if last_result is not None else frame.copy()

    # 상단 정보 오버레이 (한글)
    display = put_korean_text(display, f'분석 간격: {frame_skip}프레임 (+빠르게  -느리게)', (10, 10), (200, 200, 200), font_small)

    cv2.imshow('실시간 감정+포즈 분석 (q:종료)', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == 45:    # + 키 → 더 자주 분석
        frame_skip = max(1, frame_skip - 1)
        print(f'frame_skip: {frame_skip}')
    elif key == 61:    # - 키 → 덜 자주 분석
        frame_skip = min(30, frame_skip + 1)
        print(f'frame_skip: {frame_skip}')

camera.release()
cv2.destroyAllWindows()

앱 시작!  q: 종료
frame_skip=5 — 위/아래 방향키로 조절


### 🎯 GenAI 활용 — 파이프라인 확장

---
**프롬프트 A: 감정별 색상 테두리 굵기 조절**
```
현재 감정 인식 앱에서 감정에 따라 바운딩 박스 테두리 굵기를 다르게 해줘.
happy → 굵기 4 (강조)
anger → 굵기 4 + 박스 깜빡임 효과 (프레임마다 색상 토글)
나머지 → 굵기 2
```

**프롬프트 B: 감정 히스토리 그래프**
```
실시간 웹캠 감정 인식 앱에서
최근 50프레임의 감정 분포를 화면 오른쪽에 막대 그래프로 실시간 표시해줘.
cv2로 직접 그리고, 각 감정(neutral/happy/sad/surprise/anger)마다
다른 색상 막대를 표시해줘.
```
---

In [6]:
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
# your code here
from collections import deque
from PIL import ImageFont, ImageDraw, Image
import numpy as np
import cv2

# 한글 폰트 설정
font_path = "C:/Windows/Fonts/malgun.ttf"
font_small  = ImageFont.truetype(font_path, 20)
font_medium = ImageFont.truetype(font_path, 26)

face_input_h = face_compiled.input(0).shape[2]
face_input_w = face_compiled.input(0).shape[3]

# --- 프롬프트 B: 최근 50프레임 감정 저장을 위한 큐(Queue) 생성 ---
emotion_history = deque(maxlen=50)

def detect_faces(frame, threshold=0.5):
    h, w   = frame.shape[:2]
    f_in   = cv2.resize(frame, (face_input_w, face_input_h))
    f_in   = np.expand_dims(f_in.transpose(2,0,1), 0).astype(np.float32)
    dets   = face_compiled([f_in])[face_compiled.output(0)]
    result = []
    for det in dets[0][0]:
        if det[2] > threshold:
            x1,y1 = max(0,int(det[3]*w)), max(0,int(det[4]*h))
            x2,y2 = min(w,int(det[5]*w)), min(h,int(det[6]*h))
            result.append((x1,y1,x2,y2,float(det[2])))
    return result

def put_korean_text(img, text, position, color=(0, 255, 255), font=None):
    if font is None:
        font = font_medium
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    draw    = ImageDraw.Draw(img_pil)
    r, g, b = color[2], color[1], color[0]
    draw.text(position, text, font=font, fill=(r, g, b))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

# 실시간 감정 + 포즈 분석 앱
camera      = cv2.VideoCapture(0)
frame_skip  = 5
frame_count = 0
last_result = None

# 프롬프트 A 적용을 위해 frame_count를 인자로 받도록 수정
def analyze_frame(frame, current_frame_count):
    result = frame.copy()
    h, w   = frame.shape[:2]

    # 1. 얼굴 검출
    detections = detect_faces(frame)
    detected_emotion_this_frame = None

    for (x1, y1, x2, y2, conf) in detections:
        # 2. 감정 인식
        roi = frame[y1:y2, x1:x2]
        if roi.size > 0:
            e_in  = np.expand_dims(cv2.resize(roi, (64, 64)).transpose(2, 0, 1), 0).astype(np.float32)
            probs = emotion_compiled([e_in])[emotion_compiled.output(0)][0].flatten()
            emo   = EMOTIONS[int(np.argmax(probs))]
            base_color = EMOTION_COLOR.get(emo, (255, 255, 255))
            
            # 그래프 기록용으로 첫 번째 얼굴의 감정만 저장
            if detected_emotion_this_frame is None:
                detected_emotion_this_frame = emo

            # --- 프롬프트 A: 감정별 테두리 굵기 및 깜빡임 로직 ---
            thickness = 2
            draw_color = base_color
            
            if emo == 'happy':
                thickness = 4
            elif emo == 'anger':
                thickness = 4
                # 프레임 카운트를 이용해 색상 토글 (깜빡임 효과: 빨간색 <-> 흰색)
                if (current_frame_count // 5) % 2 == 0:
                    draw_color = (255, 255, 255) # 흰색으로 토글
            # ----------------------------------------------------

            cv2.rectangle(result, (x1, y1), (x2, y2), draw_color, thickness)
            label_y = max(0, y1 - 35)
            result  = put_korean_text(result, f'{emo} {probs.max():.0%}', (x1, label_y), base_color, font_medium)

    # 인식된 감정이 있으면 히스토리에 추가
    if detected_emotion_this_frame:
        emotion_history.append(detected_emotion_this_frame)

    # 3. 관절 포즈 추정
    pi      = pose_compiled.input(0)
    ph, pw  = pi.shape[2], pi.shape[3]
    p_in    = np.expand_dims(cv2.resize(frame, (pw, ph)).transpose(2, 0, 1), 0).astype(np.float32)
    heatmaps = pose_compiled([p_in])[pose_compiled.output(1)]

    keypoints = []
    for i in range(18):
        hm = heatmaps[0, i]
        _, conf, _, pt = cv2.minMaxLoc(hm)
        keypoints.append((int(pt[0]*w/hm.shape[1]), int(pt[1]*h/hm.shape[0]), float(conf)))

    for (p1, p2) in POSE_PAIRS:
        x1, y1, c1 = keypoints[p1]
        x2, y2, c2 = keypoints[p2]
        if c1 > 0.1 and c2 > 0.1:
            cv2.line(result, (x1, y1), (x2, y2), (0, 255, 255), 2)
    for (x, y, c) in keypoints:
        if c > 0.1:
            cv2.circle(result, (x, y), 4, (0, 200, 255), -1)

    # --- 프롬프트 B: 감정 히스토리 막대 그래프 그리기 ---
    if len(emotion_history) > 0:
        # 가독성을 위해 우측 상단에 반투명 검은색 배경(Overlay) 깔기
        overlay = result.copy()
        cv2.rectangle(overlay, (w - 180, 20), (w - 10, 220), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, result, 0.4, 0, result)

        # 그래프 제목
        cv2.putText(result, "Recent 50 Frames", (w - 170, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

        total_history = len(emotion_history)
        
        # 5가지 감정에 대해 막대 그리기
        for i, target_emo in enumerate(EMOTIONS):
            count = emotion_history.count(target_emo)
            # 50프레임 대비 비율 계산
            ratio = count / total_history
            
            # 막대바 설정 (최대 길이를 100px로 잡음)
            bar_max_width = 100
            bar_width = int(ratio * bar_max_width)
            bar_y = 70 + (i * 28) # 세로 간격 28px
            
            color = EMOTION_COLOR.get(target_emo, (255, 255, 255))
            
            # 감정 이름 첫 글자 표시 (예: H, N, S...)
            cv2.putText(result, target_emo[0].upper(), (w - 170, bar_y + 13), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            # 막대 바 그리기
            cv2.rectangle(result, (w - 145, bar_y), (w - 145 + bar_width, bar_y + 15), color, -1)
            # 막대 옆에 퍼센트 수치 표시
            cv2.putText(result, f"{int(ratio*100)}%", (w - 140 + bar_width + 5, bar_y + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
    # ----------------------------------------------------

    return result

print('앱 시작!  q: 종료')
print(f'frame_skip={frame_skip} — 위/아래 방향키로 조절')

while True:
    ret, frame = camera.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % frame_skip == 0:
        # 함수 호출 시 현재 프레임 카운트 함께 전달
        last_result = analyze_frame(frame, frame_count)

    display = last_result if last_result is not None else frame.copy()

    # 상단 정보 오버레이 (한글)
    display = put_korean_text(display, f'분석 간격: {frame_skip}프레임 (+빠르게  -느리게)', (10, 10), (200, 200, 200), font_small)

    cv2.imshow('실시간 감정+포즈 분석 (q:종료)', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == 45:    # + 키 → 더 자주 분석
        frame_skip = max(1, frame_skip - 1)
        print(f'frame_skip: {frame_skip}')
    elif key == 61:    # - 키 → 덜 자주 분석
        frame_skip = min(30, frame_skip + 1)
        print(f'frame_skip: {frame_skip}')

camera.release()
cv2.destroyAllWindows()

앱 시작!  q: 종료
frame_skip=5 — 위/아래 방향키로 조절


---
## 🎯 도전 과제 3 — Streamlit으로 웹 앱 배포하기

지금까지 만든 필터 앱은 **내 컴퓨터에서만** 실행됩니다.  
Streamlit을 사용하면 누구나 브라우저로 접속할 수 있는 **웹 앱**으로 만들 수 있습니다!

### Streamlit이란?
Python 코드만으로 웹 앱을 만들 수 있는 라이브러리입니다.  
HTML/CSS/JavaScript를 전혀 몰라도 됩니다.

```
지금까지        →       Streamlit 적용 후
cv2.imshow()          브라우저에서 실시간 확인
로컬에서만 실행        링크 공유로 누구나 접속
키보드로 조작          버튼/슬라이더로 조작
```

### 동작 방식 (웹캠 대신 이미지 업로드)
웹 환경에서는 웹캠 직접 접근이 제한되므로  
**이미지를 업로드하면 필터를 적용해서 보여주는 앱**으로 만듭니다.

```
이미지 업로드  ->  필터 선택 (사이드바)  ->  원본 vs 결과 나란히 표시
```

### Step 1 — Streamlit 설치 확인

아래 셀을 실행해서 설치 여부를 확인하세요.

In [7]:
try:
    import streamlit
    print("Streamlit 버전:", streamlit.__version__)
except ImportError:
    print("미설치. 아래 명령어로 설치하세요:")
    print("  pip install streamlit")

미설치. 아래 명령어로 설치하세요:
  pip install streamlit


### Step 2 — GenAI로 앱 코드 생성하기

아래 프롬프트를 그대로 복사해서 GenAI에게 전달하세요.  
받은 코드를 `filter_app.py` 파일로 저장합니다.

---
**프롬프트:**
```
Python Streamlit과 OpenCV로 이미지 필터 비교 웹 앱을 만들어줘.

[앱 구조]
- 사이드바에서 필터 선택 (selectbox)
- 메인 화면에 이미지 업로드 (file_uploader, jpg/png)
- 업로드한 이미지에 선택한 필터 적용
- 원본과 필터 적용 결과를 나란히(columns) 표시
- 필터 적용된 이미지 다운로드 버튼 추가

[필터 목록] — 각각 함수로 구현
1. 원본 (변환 없음)
2. 회색조 (Grayscale)
3. Gaussian 블러 (커널 크기를 슬라이더로 조정, 범위 1~31 홀수만)
4. Canny 엣지 (threshold1, threshold2를 슬라이더로 조정)
5. 세피아 (numpy 행렬 변환)
6. 선명화 (Sharpening kernel)

[기술 조건]
- OpenCV는 BGR이므로 PIL 또는 cvtColor로 RGB 변환 후 st.image() 사용
- 업로드 이미지는 np.frombuffer + cv2.imdecode로 읽기
- 다운로드 버튼은 st.download_button 사용, PNG 형식
- 파일명: filter_app.py
- 한국어 UI (제목, 레이블, 설명 모두 한국어)

[앱 제목]: "이미지 필터 비교 앱"
[사이드바 제목]: "필터 설정"
```
---

In [ ]:
# 받은 코드를 filter_app.py로 저장하는 방법

app_code = """
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
"""

with open("filter_app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("filter_app.py 저장 완료!")

### Step 3 — 로컬에서 실행하기

파일을 저장했으면 **터미널**에서 아래 명령어를 실행하세요.

```bash
streamlit run filter_app.py
```

브라우저가 자동으로 열리면서 `http://localhost:8501` 로 접속됩니다.

### Step 4 — Streamlit Cloud로 배포하기

로컬에서 잘 동작하면 인터넷에 배포해서 링크를 공유할 수 있습니다.

**배포 순서**

1. **GitHub에 코드 올리기**
```bash
# 터미널에서
git init
git add filter_app.py
git commit -m "Add filter app"
git push origin main
```

2. **requirements.txt 만들기**  
아래 프롬프트로 GenAI에게 요청하세요.

---
**프롬프트:**
```
Streamlit과 OpenCV를 사용하는 Python 앱의 requirements.txt를 만들어줘.
필요한 패키지: streamlit, opencv-python-headless, numpy, Pillow
각 패키지의 최신 안정 버전을 포함해줘.

주의: 서버 환경에서는 opencv-python 대신
opencv-python-headless를 사용해야 해.
```
---

3. **Streamlit Cloud에서 배포**
   - [share.streamlit.io](https://share.streamlit.io) 접속
   - GitHub 계정으로 로그인
   - `New app` → 저장소 선택 → `filter_app.py` 선택 → `Deploy`
   - 몇 분 후 `https://[이름].streamlit.app` 링크 생성!

### 🎯 추가 도전 — 슬라이더로 HSV 마스크 실시간 조정

---
**프롬프트:**
```
기존 filter_app.py에 새로운 필터를 추가해줘.

[추가할 필터]: "HSV 색상 마스크"
- 사이드바에 H(색조), S(채도), V(명도) 각각의 최솟값/최댓값 슬라이더 6개 추가
  - H: 0~180, S: 0~255, V: 0~255
- 슬라이더 값으로 cv2.inRange() 마스크를 만들어 원본에 적용
- 마스크 이미지와 결과 이미지를 함께 표시
- 슬라이더를 움직이면 실시간으로 결과가 바뀌어야 함

기존 필터 목록에 "HSV 색상 마스크"를 추가하고
선택했을 때만 슬라이더가 사이드바에 나타나도록 해줘.
```
---

In [ ]:
# 여기에 GenAI가 생성한 추가 코드를 붙여넣고
# filter_app.py를 업데이트한 후 다시 실행해보세요
# your code here

---
## 오늘 완성한 것

### 오전
✅ 실시간 웹캠 필터 앱 (6가지 모드)
✅ 트랙바로 Canny 파라미터 실시간 조절
✅ 웹캠 영상 녹화 기능 (r키 토글)

### 오후
✅ Day2 감정/포즈 파이프라인 실시간 연결
✅ frame_skip으로 속도 조절
✅ Streamlit 웹 앱 배포

```
1~2일차 기초   →   3일차 통합
픽셀/마스킹        실시간 필터 앱
Edge/Contour       트랙바 파라미터 조절
OpenVINO 모델      감정+포즈 실시간 분석
```

> 내일은 OpenVINO로 **Streamlit 미니 프로젝트** 완성! 🎯